In [1]:
# !pip install anndata==0.11.4
# !pip install anndata==0.7.8
# !pip install scanpy==1.11
# !pip install scanpy==1.10

In [2]:
import scanpy as sc 
import numpy as np 
from sklearn.model_selection import StratifiedShuffleSplit, ShuffleSplit, train_test_split 
from sklearn.preprocessing import LabelEncoder 
import pandas as pd 
import matplotlib.pyplot as plt

In [3]:
def load_adata(file_path):
    adata = sc.read_h5ad(file_path)
    return adata 

def map_disease_labels(adata, label_column, encoder_column): 
    encoder = LabelEncoder()  # Create a single instance of LabelEncoder 
    adata.obs[encoder_column] = encoder.fit_transform(adata.obs[label_column].values)  # Transform labels 
    mapping_disease_label = dict(zip(encoder.classes_, encoder.transform(encoder.classes_)))  # Store mapping 
    return adata, mapping_disease_label  

In [4]:
cohort_data = load_adata("../data/raw/COVID_Haniffa21/haniffa21.processed.h5ad")

In [5]:
cohort_data

AnnData object with n_obs × n_vars = 647366 × 24929
    obs: 'sample_id', 'n_genes', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'full_clustering', 'initial_clustering', 'Resample', 'Collection_Day', 'Sex', 'Age_interval', 'Swab_result', 'Status', 'Smoker', 'Status_on_day_collection', 'Status_on_day_collection_summary', 'Days_from_onset', 'Site', 'time_after_LPS', 'Worst_Clinical_Status', 'Outcome', 'patient_id'
    var: 'feature_types'
    uns: 'hvg', 'leiden', 'neighbors', 'pca', 'umap'
    obsm: 'X_pca', 'X_pca_harmony', 'X_umap'
    layers: 'raw'

In [6]:
gene_mask = cohort_data.var['feature_types'] == 'Gene Expression' 
adata_rna = cohort_data[:, gene_mask] 
cohort_data_binary = adata_rna[adata_rna.obs['Status'].isin(['Covid', 'Healthy'])] 

del cohort_data_binary.obsm 
del cohort_data_binary.uns 

In [7]:
cohort_data_binary.X = cohort_data_binary.layers['raw'] #going back to raw counts 

In [8]:
cohort_data_binary.X.max(), cohort_data_binary.X.min()

(47487.0, 0.0)

In [9]:
cohort_data_binary, mapping_disease_label  = map_disease_labels(cohort_data_binary, 'Status', 'disease_label')

In [10]:
mapping_disease_label 

{'Covid': 0, 'Healthy': 1}

In [11]:
cohort_data_binary.obs['Status'].value_counts()

Covid      527286
Healthy     97039
Name: Status, dtype: int64

In [12]:
cohort_data_binary.obs.groupby("Status")["patient_id"].nunique()

Status
Covid      90
Healthy    23
Name: patient_id, dtype: int64

In [13]:
cell_counts = cohort_data_binary.obs['patient_id'].value_counts() 

In [14]:
cell_counts

MH9143427    14317
AP6          14086
MH8919333    12081
MH9143277    11710
AP11         10921
             ...  
CV0200        1184
CV0180        1168
CV0326         784
CV0144         767
CV0944         634
Name: patient_id, Length: 113, dtype: int64

In [15]:
patients_to_keep = cell_counts[cell_counts >= 1000].index

In [16]:
cohort_data_binary = cohort_data_binary[cohort_data_binary.obs['patient_id'].isin(patients_to_keep)].copy()

In [17]:
cohort_data_binary 

AnnData object with n_obs × n_vars = 622140 × 24737
    obs: 'sample_id', 'n_genes', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'full_clustering', 'initial_clustering', 'Resample', 'Collection_Day', 'Sex', 'Age_interval', 'Swab_result', 'Status', 'Smoker', 'Status_on_day_collection', 'Status_on_day_collection_summary', 'Days_from_onset', 'Site', 'time_after_LPS', 'Worst_Clinical_Status', 'Outcome', 'patient_id', 'disease_label'
    var: 'feature_types'
    layers: 'raw'

In [18]:
cohort_data_binary.obs['patient_id']

covid_index
AAACCTGAGAAACCTA-MH9179824    MH9179824
AAACCTGAGAGTAATC-MH9179824    MH9179824
AAACCTGAGAGTGAGA-MH9179824    MH9179824
AAACCTGAGGAATCGC-MH9179824    MH9179824
AAACCTGAGTGTTGAA-MH9179824    MH9179824
                                ...    
BGCV15_TTTGGTTGTTGGGACA-1        CV0176
BGCV15_TTTGTCAAGGCGATAC-1        CV0176
BGCV15_TTTGTCACAGACACTT-1        CV0257
BGCV15_TTTGTCAGTTACGGAG-1        CV0257
BGCV15_TTTGTCATCGAATGGG-1        CV0176
Name: patient_id, Length: 622140, dtype: category
Categories (110, object): ['AP1', 'AP2', 'AP3', 'AP4', ..., 'newcastle49', 'newcastle59', 'newcastle65', 'newcastle74']

In [19]:
cohort_data_binary.obs.groupby('Status')['patient_id'].size()

Status
Covid      525735
Healthy     96405
Name: patient_id, dtype: int64

In [20]:
# Save this to the raw folder:
path = "../data/raw/COVID_Haniffa21/cohort_data.h5ad"
cohort_data_binary.write_h5ad(path)

In [21]:
ls ../data/raw/COVID_Haniffa21/

cohort_data.h5ad   cohort_node2.h5ad           haniffa21.processed.h5ad
cohort_node1.h5ad  haniffa21.max_patient.h5ad  preprocessing_covid.ipynb


In [22]:
# divide patients for two nodes 
covid_cohort = cohort_data_binary[cohort_data_binary.obs['Status'].isin(['Covid'])] 
healthy_cohort = cohort_data_binary[cohort_data_binary.obs['Status'].isin(['Healthy'])] 

In [23]:
covid_cohort.obs['Status'].unique()

['Covid']
Categories (1, object): ['Covid']

In [24]:
healthy_cohort.obs['Status'].unique()

['Healthy']
Categories (1, object): ['Healthy']

In [25]:
covid_patients=np.unique(covid_cohort.obs['patient_id'])
healthy_patients=np.unique(healthy_cohort.obs['patient_id'])

In [26]:
len(covid_patients), len(healthy_patients)

(88, 22)

In [27]:
np.random.seed(42)
covid_part1_patients = np.random.choice(covid_patients, size=len(covid_patients)//2, replace=False)
covid_part2_patients = np.setdiff1d(covid_patients, covid_part1_patients)

healthy_part1_patients = np.random.choice(healthy_patients, size=len(healthy_patients)//2, replace=False)
healthy_part2_patients = np.setdiff1d(healthy_patients, healthy_part1_patients)

In [28]:
len(covid_part1_patients), len(covid_part2_patients)

(44, 44)

In [29]:
len(healthy_part1_patients), len(healthy_part2_patients)

(11, 11)

In [30]:
covid_node1 = cohort_data_binary[cohort_data_binary.obs['patient_id'].isin(covid_part1_patients)].copy()
covid_node2 = cohort_data_binary[cohort_data_binary.obs['patient_id'].isin(covid_part2_patients)].copy()

healthy_node1 = cohort_data_binary[cohort_data_binary.obs['patient_id'].isin(healthy_part1_patients)].copy()
healthy_node2 = cohort_data_binary[cohort_data_binary.obs['patient_id'].isin(healthy_part2_patients)].copy()

In [31]:
healthy_node1, healthy_node2 

(AnnData object with n_obs × n_vars = 43731 × 24737
     obs: 'sample_id', 'n_genes', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'full_clustering', 'initial_clustering', 'Resample', 'Collection_Day', 'Sex', 'Age_interval', 'Swab_result', 'Status', 'Smoker', 'Status_on_day_collection', 'Status_on_day_collection_summary', 'Days_from_onset', 'Site', 'time_after_LPS', 'Worst_Clinical_Status', 'Outcome', 'patient_id', 'disease_label'
     var: 'feature_types'
     layers: 'raw',
 AnnData object with n_obs × n_vars = 52674 × 24737
     obs: 'sample_id', 'n_genes', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'full_clustering', 'initial_clustering', 'Resample', 'Collection_Day', 'Sex', 'Age_interval', 'Swab_result', 'Status', 'Smoker', 'Status_on_day_collection', 'Status_on_day_collection_summary', 'Days_from_onset', 'Site', 'time_after_LPS', 'Worst_Clinical_Status', 'Outcome', 'patient_id', 'disease_label'
     var: 'feature_types'
  

In [32]:
covid_node1.obs['Status'].value_counts(), covid_node2.obs['Status'].value_counts() 

(Covid    232829
 Name: Status, dtype: int64,
 Covid    292906
 Name: Status, dtype: int64)

In [33]:
healthy_node1.obs['Status'].value_counts(), healthy_node2.obs['Status'].value_counts() 

(Healthy    43731
 Name: Status, dtype: int64,
 Healthy    52674
 Name: Status, dtype: int64)

In [34]:
healthy_node1.obs['patient_id'].unique()

['MH8919332', 'MH8919227', 'MH8919176', 'MH8919179', 'MH8919282', ..., 'CV0911', 'CV0915', 'CV0917', 'CV0926', 'CV0940']
Length: 11
Categories (11, object): ['CV0904', 'CV0911', 'CV0915', 'CV0917', ..., 'MH8919179', 'MH8919227', 'MH8919282', 'MH8919332']

In [35]:
healthy_node2.obs['patient_id'].unique()

['newcastle65', 'MH8919226', 'MH8919333', 'MH8919283', 'MH8919178', ..., 'newcastle74', 'CV0902', 'CV0929', 'CV0939', 'CV0934']
Length: 11
Categories (11, object): ['CV0902', 'CV0929', 'CV0934', 'CV0939', ..., 'MH8919283', 'MH8919333', 'newcastle65', 'newcastle74']

In [36]:
import anndata as ad
node1 = ad.concat([covid_node1, healthy_node1], join="inner")
node2 = ad.concat([covid_node2, healthy_node2], join="inner")

In [37]:
node1

AnnData object with n_obs × n_vars = 276560 × 24737
    obs: 'sample_id', 'n_genes', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'full_clustering', 'initial_clustering', 'Resample', 'Collection_Day', 'Sex', 'Age_interval', 'Swab_result', 'Status', 'Smoker', 'Status_on_day_collection', 'Status_on_day_collection_summary', 'Days_from_onset', 'Site', 'time_after_LPS', 'Worst_Clinical_Status', 'Outcome', 'patient_id', 'disease_label'
    layers: 'raw'

In [38]:
node2

AnnData object with n_obs × n_vars = 345580 × 24737
    obs: 'sample_id', 'n_genes', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'full_clustering', 'initial_clustering', 'Resample', 'Collection_Day', 'Sex', 'Age_interval', 'Swab_result', 'Status', 'Smoker', 'Status_on_day_collection', 'Status_on_day_collection_summary', 'Days_from_onset', 'Site', 'time_after_LPS', 'Worst_Clinical_Status', 'Outcome', 'patient_id', 'disease_label'
    layers: 'raw'

In [39]:
node1.obs.groupby("Status")["patient_id"].nunique()

Status
Covid      44
Healthy    11
Name: patient_id, dtype: int64

In [40]:
node2.obs.groupby("Status")["patient_id"].nunique()

Status
Covid      44
Healthy    11
Name: patient_id, dtype: int64

In [41]:
set(node1.obs['patient_id'].unique()) & set(node2.obs['patient_id'].unique())

set()

In [42]:
node1.X.max(), node2.X.max()

(33151.0, 47487.0)

In [43]:
path = "../data/raw/COVID_Haniffa21/cohort_node1.h5ad"
node1.write_h5ad(path)

/usr/local/lib/python3.9/site-packages/anndata/_core/anndata.py:1228: FutureWarning: The `inplace` parameter in pandas.Categorical.reorder_categories is deprecated and will be removed in a future version. Removing unused categories will always return a new Categorical object.
  c.reorder_categories(natsorted(c.categories), inplace=True)
... storing 'sample_id' as categorical
/usr/local/lib/python3.9/site-packages/anndata/_core/anndata.py:1228: FutureWarning: The `inplace` parameter in pandas.Categorical.reorder_categories is deprecated and will be removed in a future version. Removing unused categories will always return a new Categorical object.
  c.reorder_categories(natsorted(c.categories), inplace=True)
... storing 'full_clustering' as categorical
/usr/local/lib/python3.9/site-packages/anndata/_core/anndata.py:1228: FutureWarning: The `inplace` parameter in pandas.Categorical.reorder_categories is deprecated and will be removed in a future version. Removing unused categories will a

In [44]:
path = "../data/raw/COVID_Haniffa21/cohort_node2.h5ad"
node2.write_h5ad(path)

/usr/local/lib/python3.9/site-packages/anndata/_core/anndata.py:1228: FutureWarning: The `inplace` parameter in pandas.Categorical.reorder_categories is deprecated and will be removed in a future version. Removing unused categories will always return a new Categorical object.
  c.reorder_categories(natsorted(c.categories), inplace=True)
... storing 'sample_id' as categorical
/usr/local/lib/python3.9/site-packages/anndata/_core/anndata.py:1228: FutureWarning: The `inplace` parameter in pandas.Categorical.reorder_categories is deprecated and will be removed in a future version. Removing unused categories will always return a new Categorical object.
  c.reorder_categories(natsorted(c.categories), inplace=True)
... storing 'full_clustering' as categorical
/usr/local/lib/python3.9/site-packages/anndata/_core/anndata.py:1228: FutureWarning: The `inplace` parameter in pandas.Categorical.reorder_categories is deprecated and will be removed in a future version. Removing unused categories will a

In [45]:
print("done")

done
